In [ ]:
"""
Combines fcm_recalculation.py and mm_fcm_refit.py into one script.

Outputs all thesis numbers and saves results to OUT_DIR.

FCM = 0.4 * milk_kg + 15 * fat_kg  (NRC 2001)
NRC DMI = (0.372*FCM + 0.0968*BW^0.75) * (1 - exp(-0.192*(DIM/7+3.67)))
"""

import numpy as np
import pandas as pd
import os
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# PATHS

CSV_001       = "/Users/shreyarao/Desktop/MooOutput/001inputs/"
CSV_002       = "/Users/shreyarao/Desktop/MooOutput/002inputs/"
CSV_003       = "/Users/shreyarao/Desktop/MooOutput/003inputs/"
MILK_XLS_PATH = "/Users/shreyarao/Desktop/MooOutput/Copy of milk yield.xls"
OUT_DIR       = "/Users/shreyarao/Desktop/MooOutput/nutrition_results/"
os.makedirs(OUT_DIR, exist_ok=True)

DAYS = ["March15","March16","March17","March18","March19",
        "March20","March21","March22","March23","March24","March25"]


# COW PARAMETERS

COW_BASE = {
    387: {"BW":921.0, "DIM_mar15":15,  "fat_pct":5.25, "protein_pct":3.43, "parity":4, "camera":"001"},
    408: {"BW":777.0, "DIM_mar15":354, "fat_pct":5.32, "protein_pct":4.06, "parity":2, "camera":"001"},
    410: {"BW":623.0, "DIM_mar15":159, "fat_pct":4.67, "protein_pct":3.72, "parity":2, "camera":"001"},
    416: {"BW":709.0, "DIM_mar15":196, "fat_pct":4.86, "protein_pct":4.20, "parity":2, "camera":"001"},
    426: {"BW":801.0, "DIM_mar15":245, "fat_pct":4.77, "protein_pct":3.45, "parity":1, "camera":"002"},
    413: {"BW":680.0, "DIM_mar15":224, "fat_pct":5.01, "protein_pct":3.89, "parity":2, "camera":"002"},
    323: {"BW":801.0, "DIM_mar15":289, "fat_pct":4.98, "protein_pct":3.80, "parity":6, "camera":"002"},
    354: {"BW":880.0, "DIM_mar15":214, "fat_pct":5.37, "protein_pct":4.01, "parity":5, "camera":"002"},
    310: {"BW":840.0, "DIM_mar15":61,  "fat_pct":3.99, "protein_pct":3.22, "parity":7, "camera":"002"},
    349: {"BW":921.0, "DIM_mar15":290, "fat_pct":4.66, "protein_pct":3.99, "parity":5, "camera":"002"},
    433: {"BW":657.0, "DIM_mar15":74,  "fat_pct":4.11, "protein_pct":3.48, "parity":1, "camera":"002"},
    406: {"BW":840.0, "DIM_mar15":231, "fat_pct":4.60, "protein_pct":3.28, "parity":2, "camera":"003"},
    386: {"BW":764.0, "DIM_mar15":327, "fat_pct":5.05, "protein_pct":3.89, "parity":3, "camera":"003"},
    412: {"BW":801.0, "DIM_mar15":226, "fat_pct":4.59, "protein_pct":3.96, "parity":2, "camera":"003"},
    384: {"BW":764.0, "DIM_mar15":43,  "fat_pct":4.99, "protein_pct":3.72, "parity":4, "camera":"003"},
    428: {"BW":623.0, "DIM_mar15":150, "fat_pct":4.48, "protein_pct":3.68, "parity":1, "camera":"003"},
}

STALL_TO_COW_002 = {1:426, 2:413, 3:323, 4:354, 5:310, 6:349, 7:433}

PARTIAL_DAYS = {
    (387,"March25"),(403,"March24"),
    (387,"March20"),(403,"March20"),(408,"March20"),(410,"March20"),
    (416,"March20"),(433,"March20"),(412,"March20"),(426,"March20"),
    (323,"March20"),
    (354,"March20"),(354,"March21"),(354,"March22"),
    (354,"March23"),(354,"March24"),(354,"March25"),
    (310,"March19"),(428,"March19"),
}


# FUNCTIONS

def calc_fcm(milk_kg, fat_pct):
    return 0.4 * milk_kg + 15 * (milk_kg * fat_pct / 100)

def nrc_dmi(fcm, bw, dim):
    return (0.372 * fcm + 0.0968 * (bw**0.75)) * (1 - np.exp(-0.192*(dim/7+3.67)))

def mape(yt, yp):
    yt, yp = np.array(yt), np.array(yp)
    return np.mean(np.abs((yt-yp)/np.clip(np.abs(yt),1e-8,None)))*100

def report(yt, yp, label):
    yt, yp = np.array(yt), np.array(yp)
    rmse = np.sqrt(mean_squared_error(yt,yp))
    mae  = mean_absolute_error(yt,yp)
    bias = np.mean(yp-yt)
    mp   = mape(yt,yp)
    r2   = r2_score(yt,yp)
    print(f"  {label}: RMSE={rmse:.3f}  MAE={mae:.3f}  Bias={bias:+.3f}  MAPE={mp:.1f}%  R2={r2:.3f}")
    return {"label":label,"RMSE":round(rmse,3),"MAE":round(mae,3),
            "Bias":round(bias,3),"MAPE":round(mp,1),"R2":round(r2,3),"N":len(yt)}

def fit_reg(df):
    X = np.column_stack([np.ones(len(df)), df["feeding_min"], df["BW"], df["FCM_kg"]])
    c,_,_,_ = np.linalg.lstsq(X, df["DMI_NRC"].values, rcond=None)
    return c

def pred_reg(df, c):
    X = np.column_stack([np.ones(len(df)), df["feeding_min"], df["BW"], df["FCM_kg"]])
    return X @ c


# DATA LOADING

FEEDING_LABELS = {"Feeding & Standing", "Feeding & Lying"}

def load_001(folder, days):
    dfs = []
    for day in days:
        p = os.path.join(folder, f"{day}_activity_totals.csv")
        if os.path.exists(p):
            df = pd.read_csv(p)
            if "day" not in df.columns: df["day"] = day
            df["camera"] = "001"
            dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def load_folder(folder, days, stall_map, label):
    dfs = []
    for day in days:
        p = os.path.join(folder, day, "cow_activity_totals.csv")
        if os.path.exists(p):
            df = pd.read_csv(p)
            df["day"] = day
            if stall_map:
                df["cow_id"] = df["cow_id"].map(stall_map)
                df = df.dropna(subset=["cow_id"])
                df["cow_id"] = df["cow_id"].astype(int)
            df["camera"] = label
            dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print("Loading data...")
df_all = pd.concat([
    load_001(CSV_001, DAYS),
    load_folder(CSV_002, DAYS, STALL_TO_COW_002, "002"),
    load_folder(CSV_003, DAYS, None, "003"),
], ignore_index=True)

xls = pd.read_excel(MILK_XLS_PATH, engine="xlrd")
xls["Date"] = pd.to_datetime(xls["Date"]).dt.date
xls["day"]  = xls["Date"].apply(lambda d: f"March{d.day}" if d.month==3 else None)
xls = xls.dropna(subset=["day"])
xls = xls[~xls["Milk duration (mm:ss)"].astype(str).str.startswith("30:00")]

MILK = {}
for _, r in (xls[xls["Animal Number"].isin(list(COW_BASE.keys()))]
             .groupby(["Animal Number","day"])["Yield"]
             .sum().reset_index()).iterrows():
    c = int(r["Animal Number"])
    if c not in MILK: MILK[c] = {}
    MILK[c][r["day"]] = round(r["Yield"], 2)


# BUILD DATASET

feeds = df_all[df_all["activity"].isin(FEEDING_LABELS)]
daily_feed = (feeds.groupby(["cow_id","day","camera"])["duration_min"]
              .sum().reset_index())
daily_feed.columns = ["cow_id","day","camera","feeding_min"]

rows = []
for _, r in daily_feed.iterrows():
    cow,day,fmin,cam = int(r["cow_id"]),r["day"],r["feeding_min"],r["camera"]
    if cow not in COW_BASE: continue
    if day not in MILK.get(cow,{}): continue
    if fmin < 1: continue
    if (cow,day) in PARTIAL_DAYS: continue
    b   = COW_BASE[cow]
    dim = b["DIM_mar15"] + DAYS.index(day)
    milk= MILK[cow][day]
    fcm = calc_fcm(milk, b["fat_pct"])
    rows.append({"cow_id":cow,"day":day,"camera":cam,"parity":b["parity"],
                 "feeding_min":round(fmin,1),"BW":b["BW"],"DIM":dim,
                 "milk_kg":round(milk,2),"FCM_kg":round(fcm,2),
                 "DMI_NRC":round(nrc_dmi(fcm,b["BW"],dim),3)})

data = pd.DataFrame(rows)
data = data[data["feeding_min"] >= 50].copy()
all_cows = sorted(data["cow_id"].unique())
cow_nrc_mean  = data.groupby("cow_id")["DMI_NRC"].mean()
cow_fmin_mean = data.groupby("cow_id")["feeding_min"].mean()
cam_map = data.drop_duplicates("cow_id").set_index("cow_id")["camera"]

print(f"Dataset: {len(data)} cow-days, {data['cow_id'].nunique()} cows")
print(f"DMI_NRC FCM range: {data['DMI_NRC'].min():.1f} - {data['DMI_NRC'].max():.1f} kg/day")
print(f"Mean DMI_NRC: {data['DMI_NRC'].mean():.2f} kg/day")


# DAY-SPLIT PARTITION

split_records = []
for cow in all_cows:
    s = data[data["cow_id"]==cow].copy()
    s["day_idx"] = s["day"].apply(lambda d: DAYS.index(d))
    s = s.sort_values("day_idx").reset_index(drop=True)
    n = len(s)
    if n < 4: continue
    split_pt = n // 2
    s["split"] = ["train"]*split_pt + ["test"]*(n-split_pt)
    split_records.append(s)
data_split = pd.concat(split_records, ignore_index=True)

In [ ]:
# 1. LINEAR BASELINE

print("\n" + "="*60)
print("1. LINEAR BASELINE")
print("="*60)

fmin_all = data["feeding_min"].values
y_all    = data["DMI_NRC"].values
k_local  = np.sum(fmin_all * y_all) / np.sum(fmin_all**2)
report(y_all, k_local*fmin_all, f"Local k={k_local:.4f} whole-dataset")

k_lit = 27.0/230.4
report(y_all, k_lit*fmin_all, f"Literature k={k_lit:.4f} whole-dataset")

# Day-split
tr_lin = data_split[data_split["split"]=="train"]
te_lin = data_split[data_split["split"]=="test"].copy()
k_tr   = np.sum(tr_lin["feeding_min"]*tr_lin["DMI_NRC"])/np.sum(tr_lin["feeding_min"]**2)
te_lin["DMI_est"] = k_tr * te_lin["feeding_min"]
lin_test = report(te_lin["DMI_NRC"], te_lin["DMI_est"], "Linear day-split TEST")

# LOCO
loco_lin = []
for ho in all_cows:
    tr = data[data["cow_id"]!=ho]
    te = data[data["cow_id"]==ho]
    k  = np.sum(tr["feeding_min"]*tr["DMI_NRC"])/np.sum(tr["feeding_min"]**2)
    pred = k*te["feeding_min"].values
    for i,(_,row) in enumerate(te.iterrows()):
        loco_lin.append({"cow_id":ho,"DMI_NRC":row["DMI_NRC"],"DMI_est":pred[i]})
loco_lin_df = pd.DataFrame(loco_lin)
lin_loco = report(loco_lin_df["DMI_NRC"], loco_lin_df["DMI_est"], "Linear LOCO")

In [ ]:
# 2. POOLED REGRESSION

print("\n" + "="*60)
print("2. POOLED REGRESSION  DMI = a + b*f_feed + c*BW + d*FCM")
print("="*60)

coeffs_all = fit_reg(data)
print(f"\n  Whole-dataset: a={coeffs_all[0]:.4f}  b={coeffs_all[1]:.5f}  "
      f"c={coeffs_all[2]:.6f}  d={coeffs_all[3]:.5f}")

tr_reg = data_split[data_split["split"]=="train"].copy()
te_reg = data_split[data_split["split"]=="test"].copy()
cf_tr  = fit_reg(tr_reg)
tr_reg["DMI_est"] = pred_reg(tr_reg, cf_tr)
te_reg["DMI_est"] = pred_reg(te_reg, cf_tr)
report(tr_reg["DMI_NRC"], tr_reg["DMI_est"], "Regression day-split TRAIN")
reg_test = report(te_reg["DMI_NRC"], te_reg["DMI_est"], "Regression day-split TEST")

print(f"\n  Per-cow LOCO:")
print(f"  {'Cow':>5}  {'n':>3}  {'a':>7}  {'b':>9}  {'c':>9}  {'d':>9}  {'RMSE':>6}  {'MAPE':>6}")
print("  "+"-"*70)
loco_reg = []
for ho in all_cows:
    tr  = data[data["cow_id"]!=ho]
    te  = data[data["cow_id"]==ho]
    cf  = fit_reg(tr)
    pred= pred_reg(te,cf)
    err = pred-te["DMI_NRC"].values
    pct = 100*err/te["DMI_NRC"].values
    rmse= np.sqrt((err**2).mean())
    mp  = np.abs(pct).mean()
    print(f"  {ho:>5}  {len(te):>3}  {cf[0]:>7.3f}  {cf[1]:>9.5f}  "
          f"{cf[2]:>9.6f}  {cf[3]:>9.5f}  {rmse:>6.3f}  {mp:>5.1f}%")
    for i,(_,row) in enumerate(te.iterrows()):
        loco_reg.append({"cow_id":ho,"DMI_NRC":row["DMI_NRC"],"DMI_est":pred[i]})
loco_reg_df = pd.DataFrame(loco_reg)
reg_loco = report(loco_reg_df["DMI_NRC"], loco_reg_df["DMI_est"], "Regression LOCO pooled")

In [ ]:
# 3. MICHAELIS-MENTEN — CALIBRATED DMI_MAX

print("\n" + "="*60)
print("3. MICHAELIS-MENTEN MODEL")
print("="*60)

# Shared DMI_MAX (for comparison row in Table 6.8)
shared_max = data["DMI_NRC"].mean()
est_shared = shared_max * data["feeding_min"] / (30 + data["feeding_min"])
report(data["DMI_NRC"], est_shared, f"M-M shared DMI_MAX={shared_max:.2f}, Km=30")

# Km grid search with analytically calibrated per-cow DMI_MAX
print(f"\n  Km sensitivity sweep (analytically calibrated per-cow DMI_MAX):")
print(f"  {'Km':>6}  {'RMSE':>6}  {'MAE':>6}  {'MAPE':>6}  {'Bias':>7}  {'R2':>6}  {'Within10%':>10}")
print("  "+"-"*65)

KM_GRID = [10,20,30,40,50,60,70,80,90,100,120,150,200,300]
sweep = []
for km in KM_GRID:
    dmi_max_i = cow_nrc_mean * (km + cow_fmin_mean) / cow_fmin_mean
    data["DMI_MAX_i"] = data["cow_id"].map(dmi_max_i)
    est = data.apply(lambda r: r["DMI_MAX_i"]*r["feeding_min"]/(km+r["feeding_min"]), axis=1)
    err = est - data["DMI_NRC"]
    rmse= np.sqrt((err**2).mean())
    mae = err.abs().mean()
    bias= err.mean()
    mp  = mape(data["DMI_NRC"],est)
    r2  = r2_score(data["DMI_NRC"],est)
    w10 = ((est/data["DMI_NRC"]).between(0.9,1.1)).sum()
    print(f"  {km:>6}  {rmse:>6.3f}  {mae:>6.3f}  {mp:>5.1f}%  {bias:>+7.3f}  {r2:>6.3f}  {w10:>4}/{len(data)}")
    sweep.append({"Km":km,"RMSE":rmse,"MAE":mae,"MAPE":mp,"Bias":bias,"R2":r2,"Within10":w10})

sweep_df = pd.DataFrame(sweep)
bio = sweep_df[sweep_df["Km"].between(30,150)]
best = bio.loc[bio["MAPE"].idxmin()]
KM_BEST = int(best["Km"])
print(f"\n  Best Km (30-150): Km={KM_BEST}  MAPE={best['MAPE']:.1f}%  RMSE={best['RMSE']:.3f}  R2={best['R2']:.3f}")

# Final M-M with best Km
dmi_max_final = cow_nrc_mean * (KM_BEST + cow_fmin_mean) / cow_fmin_mean
data["DMI_MAX_i"] = data["cow_id"].map(dmi_max_final)
data["DMI_est_mm"] = data.apply(
    lambda r: r["DMI_MAX_i"]*r["feeding_min"]/(KM_BEST+r["feeding_min"]), axis=1)
data["err_mm"]   = data["DMI_est_mm"] - data["DMI_NRC"]
data["ratio_mm"] = data["DMI_est_mm"] / data["DMI_NRC"]
w10_final = (data["ratio_mm"].between(0.9,1.1)).sum()
mm_whole = report(data["DMI_NRC"], data["DMI_est_mm"], f"M-M whole dataset (Km={KM_BEST})")
print(f"  Within 10%: {w10_final}/{len(data)}")

print(f"\n  Per-cow DMI_MAX (Km={KM_BEST}):")
for cow in all_cows:
    print(f"    Cow {cow}: DMI_MAX={dmi_max_final[cow]:.2f}  NRC_mean={cow_nrc_mean[cow]:.2f}")

# Day-split M-M
mm_split = []
for cow in all_cows:
    s  = data_split[data_split["cow_id"]==cow]
    tr = s[s["split"]=="train"]
    te = s[s["split"]=="test"]
    if len(tr)==0 or len(te)==0: continue
    dmi_max_tr = cow_nrc_mean[cow]*(KM_BEST+cow_fmin_mean[cow])/cow_fmin_mean[cow]
    for _,row in te.iterrows():
        est = dmi_max_tr*row["feeding_min"]/(KM_BEST+row["feeding_min"])
        mm_split.append({"cow_id":cow,"split":"test","DMI_NRC":row["DMI_NRC"],"DMI_est":est})
    for _,row in tr.iterrows():
        est = dmi_max_tr*row["feeding_min"]/(KM_BEST+row["feeding_min"])
        mm_split.append({"cow_id":cow,"split":"train","DMI_NRC":row["DMI_NRC"],"DMI_est":est})

mm_split_df = pd.DataFrame(mm_split)
mm_tr = mm_split_df[mm_split_df["split"]=="train"]
mm_te = mm_split_df[mm_split_df["split"]=="test"]
mm_train_res = report(mm_tr["DMI_NRC"], mm_tr["DMI_est"], f"M-M day-split TRAIN")
mm_test_res  = report(mm_te["DMI_NRC"], mm_te["DMI_est"], f"M-M day-split TEST")

# Health event verification
print(f"\n  Health event verification:")
cow354_mar19 = data[(data["cow_id"]==354)&(data["day"]=="March19")]
if len(cow354_mar19)>0:
    r = cow354_mar19.iloc[0]
    print(f"    Cow 354 March19: feeding={r['feeding_min']:.1f}min  "
          f"DMI_est={r['DMI_est_mm']:.2f}  DMI_NRC={r['DMI_NRC']:.2f}  "
          f"ratio={r['ratio_mm']:.3f}")

In [ ]:
# 4. ABLATION

print("\n" + "="*60)
print("4. ABLATION — INCREMENTAL VALUE OF FEEDING DURATION")
print("="*60)

abl_models = {
    "1_PopMean":    [],
    "2_Physiology": ["BW","FCM_kg","DIM"],
    "3_Behaviour":  ["feeding_min"],
    "4_Phys_Behav": ["BW","FCM_kg","DIM","feeding_min"],
}
abl_results = {n:{"y_pred":[],"y_true":[]} for n in abl_models}
for ho in all_cows:
    tr  = data[data["cow_id"]!=ho]
    te  = data[data["cow_id"]==ho]
    yt  = te["DMI_NRC"].values
    ytr = tr["DMI_NRC"].values
    for mn,feats in abl_models.items():
        if mn=="1_PopMean":
            pred = np.full(len(yt), ytr.mean())
        else:
            pred = LinearRegression().fit(tr[feats].values,ytr).predict(te[feats].values)
        abl_results[mn]["y_pred"].extend(pred.tolist())
        abl_results[mn]["y_true"].extend(yt.tolist())

print(f"\n  {'Model':<20}  {'N':>5}  {'RMSE':>6}  {'MAE':>6}  {'MAPE':>6}  {'R2':>6}")
print("  "+"-"*55)
abl_summary = []
for mn in abl_models:
    yt = np.array(abl_results[mn]["y_true"])
    yp = np.array(abl_results[mn]["y_pred"])
    rmse= np.sqrt(mean_squared_error(yt,yp))
    mae = mean_absolute_error(yt,yp)
    mp  = mape(yt,yp)
    r2  = r2_score(yt,yp)
    print(f"  {mn:<20}  {len(yt):>5}  {rmse:>6.3f}  {mae:>6.3f}  {mp:>5.1f}%  {r2:>6.3f}")
    abl_summary.append({"Model":mn,"RMSE":round(rmse,3),"MAE":round(mae,3),
                        "MAPE":round(mp,1),"R2":round(r2,3)})

# F-test
X_p = data[["BW","FCM_kg","DIM"]].values
X_f = data[["BW","FCM_kg","DIM","feeding_min"]].values
y   = data["DMI_NRC"].values
n   = len(y)
rp  = LinearRegression().fit(X_p,y)
rf  = LinearRegression().fit(X_f,y)
ss_p= np.sum((y-rp.predict(X_p))**2)
ss_f= np.sum((y-rf.predict(X_f))**2)
kp,kf = X_p.shape[1]+1, X_f.shape[1]+1
F   = ((ss_p-ss_f)/(kf-kp))/(ss_f/(n-kf))
pF  = 1-stats.f.cdf(F,kf-kp,n-kf)
X_aug = np.column_stack([np.ones(n),X_f])
cov   = np.linalg.inv(X_aug.T@X_aug)*(ss_f/(n-kf))
se    = np.sqrt(cov[4,4])
cf    = rf.coef_[3]
print(f"\n  F({kf-kp},{n-kf})={F:.3f}  p={pF:.4f}")
print(f"  Feeding coeff: {cf:.5f} kg/min  SE={se:.5f}  95%CI=[{cf-1.96*se:.5f},{cf+1.96*se:.5f}]")

In [ ]:
# FINAL THESIS SUMMARY

print("\n" + "="*60)
print("THESIS NUMBER SUMMARY — all Chapter 6 values")
print("="*60)

print(f"\n--- LINEAR BASELINE ---")
print(f"  k (local)      = {k_local:.4f} kg/min")
print(f"  k (literature) = {k_lit:.4f} kg/min (Johnston & DeVries 2018)")
print(f"  Local k, whole dataset:  RMSE=5.366  MAE=4.179  Bias=-0.911  MAPE=15.2%")
print(f"  Literature k, whole dataset: RMSE=11.607  MAE=10.255  Bias=+9.181  MAPE=36.9%")
print(f"  Linear day-split TEST:   RMSE={lin_test['RMSE']}  MAPE={lin_test['MAPE']}%  R2={lin_test['R2']}")
print(f"  Linear LOCO:             RMSE={lin_loco['RMSE']}  MAPE={lin_loco['MAPE']}%  R2={lin_loco['R2']}")

print(f"\n--- POOLED REGRESSION ---")
print(f"  Equation: DMI = {coeffs_all[0]:.4f} + {coeffs_all[1]:.5f}*f_feed + {coeffs_all[2]:.6f}*BW + {coeffs_all[3]:.5f}*FCM")
print(f"  Day-split TRAIN: RMSE={report(tr_reg['DMI_NRC'],tr_reg['DMI_est'],'')[' RMSE'] if False else '1.506'}  MAPE=4.2%  R2=0.640")
print(f"  Day-split TEST:  RMSE={reg_test['RMSE']}  MAPE={reg_test['MAPE']}%  R2={reg_test['R2']}")
print(f"  LOCO:            RMSE={reg_loco['RMSE']}  MAPE={reg_loco['MAPE']}%  R2={reg_loco['R2']}")

print(f"\n--- M-M MODEL (Km={KM_BEST}, calibrated DMI_MAX) ---")
print(f"  Whole dataset:   RMSE={mm_whole['RMSE']}  MAPE={mm_whole['MAPE']}%  R2={mm_whole['R2']}  Within10%={w10_final}/{len(data)}")
print(f"  Day-split TRAIN: RMSE={mm_train_res['RMSE']}  MAPE={mm_train_res['MAPE']}%  R2={mm_train_res['R2']}")
print(f"  Day-split TEST:  RMSE={mm_test_res['RMSE']}  MAPE={mm_test_res['MAPE']}%  R2={mm_test_res['R2']}")

print(f"\n--- ABLATION ---")
for s in abl_summary:
    print(f"  {s['Model']:<20}: RMSE={s['RMSE']}  MAE={s['MAE']}  MAPE={s['MAPE']}%  R2={s['R2']}")
print(f"  F({kf-kp},{n-kf})={F:.3f}  p={pF:.4f}")
print(f"  Feeding coeff: +{cf:.5f} kg/min  95%CI=[{cf-1.96*se:.5f},{cf+1.96*se:.5f}]")

print(f"\n--- NRC RANGE ---")
print(f"  FCM-based DMI_NRC: {data['DMI_NRC'].min():.1f} – {data['DMI_NRC'].max():.1f} kg/day")

In [ ]:
# SAVE

data.to_csv(os.path.join(OUT_DIR,"dataset_fcm_full.csv"), index=False)
pd.DataFrame(abl_summary).to_csv(os.path.join(OUT_DIR,"ablation_results.csv"), index=False)
loco_reg_df.to_csv(os.path.join(OUT_DIR,"loco_regression.csv"), index=False)
loco_lin_df.to_csv(os.path.join(OUT_DIR,"loco_linear.csv"), index=False)
sweep_df.to_csv(os.path.join(OUT_DIR,"km_sensitivity.csv"), index=False)

print(f"\nAll saved to: {OUT_DIR}")
print("Done.")

Loading data...
Dataset: 153 cow-days, 16 cows
DMI_NRC FCM range: 17.9 - 37.3 kg/day
Mean DMI_NRC: 28.24 kg/day

1. LINEAR BASELINE
  Local k=0.0856 whole-dataset: RMSE=5.366  MAE=4.179  Bias=-0.911  MAPE=15.2%  R2=-2.547
  Literature k=0.1172 whole-dataset: RMSE=11.607  MAE=10.255  Bias=+9.181  MAPE=36.9%  R2=-15.595
  Linear day-split TEST: RMSE=5.773  MAE=4.456  Bias=-1.025  MAPE=16.1%  R2=-2.428
  Linear LOCO: RMSE=5.539  MAE=4.346  Bias=-0.872  MAPE=15.8%  R2=-2.779

2. POOLED REGRESSION  DMI = a + b*f_feed + c*BW + d*FCM

  Whole-dataset: a=3.9918  b=0.02027  c=0.013543  d=0.17852
  Regression day-split TRAIN: RMSE=1.506  MAE=1.180  Bias=+0.000  MAPE=4.2%  R2=0.640
  Regression day-split TEST: RMSE=1.811  MAE=1.377  Bias=-0.196  MAPE=4.9%  R2=0.663

  Per-cow LOCO:
    Cow    n        a          b          c          d    RMSE    MAPE
  ----------------------------------------------------------------------
    310   10    3.471    0.02092   0.013913    0.17955   0.681    1.6%
   

In [3]:

# CHAPTER 6 FIGURES 

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
import itertools

matplotlib.rcdefaults()
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "figure.dpi": 300,
})

FIG_DIR = OUT_DIR  # saves to same nutrition_results/ folder
os.makedirs(FIG_DIR, exist_ok=True)

COW_COLORS = {
    387:"#1D9E75", 408:"#BA7517", 410:"#378ADD", 416:"#D4537E",
    310:"#E24B4A", 323:"#F5A623", 349:"#9B59B6", 354:"#1ABC9C",
    413:"#2ECC71", 426:"#3498DB", 433:"#E67E22",
    406:"#C0392B", 386:"#16A085", 412:"#D35400", 384:"#2980B9", 428:"#27AE60",
}

DAY_LABELS = ["Mar 15","Mar 16","Mar 17","Mar 18","Mar 19",
              "Mar 20","Mar 21","Mar 22","Mar 23","Mar 24","Mar 25"]


# Regression day-split scatter (train vs test)
# Rebuild train/test predictions for plotting
tr_reg_plot = data_split[data_split["split"]=="train"].copy()
te_reg_plot = data_split[data_split["split"]=="test"].copy()
cf_plot = fit_reg(tr_reg_plot)
tr_reg_plot["DMI_est"] = pred_reg(tr_reg_plot, cf_plot)
te_reg_plot["DMI_est"] = pred_reg(te_reg_plot, cf_plot)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, df, title in [(axes[0], tr_reg_plot, "TRAIN (fitted days)"),
                      (axes[1], te_reg_plot, "TEST (held-out days)")]:
    mn = min(df["DMI_NRC"].min(), df["DMI_est"].min()) - 1
    mx = max(df["DMI_NRC"].max(), df["DMI_est"].max()) + 2
    ax.plot([mn,mx],[mn,mx], "k--", lw=1.2, alpha=0.6, label="Perfect agreement")
    ax.fill_between([mn,mx],[mn*0.9,mx*0.9],[mn*1.1,mx*1.1],
                    alpha=0.07, color="green", label="±10% band")
    for cow in sorted(df["cow_id"].unique()):
        s = df[df["cow_id"]==cow]
        ax.scatter(s["DMI_NRC"], s["DMI_est"],
                   color=COW_COLORS.get(cow,"gray"), s=60, alpha=0.85,
                   label=f"Cow {cow}")
    rmse_s = np.sqrt(((df["DMI_est"]-df["DMI_NRC"])**2).mean())
    mape_s = (100*(df["DMI_est"]-df["DMI_NRC"]).abs()/df["DMI_NRC"]).mean()
    r2_s   = r2_score(df["DMI_NRC"], df["DMI_est"])
    ax.set_xlabel("NRC Reference DMI (kg/day)")
    ax.set_ylabel("Estimated DMI (kg/day)")
    ax.set_title(f"{title}\nRMSE={rmse_s:.3f}  MAPE={mape_s:.1f}%  R²={r2_s:.3f}  n={len(df)}")
    ax.legend(fontsize=7, ncol=2, framealpha=0.7)
    ax.grid(alpha=0.3)
plt.suptitle("Pooled Regression Model: Train vs Held-Out Test\n"
             "DMI = a + b·f_feed + c·BW + d·FCM",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,"1regression_train_test.png"),
            dpi=300, bbox_inches="tight")
plt.close()
print("Saved")


# LOCO regression scatter


fig, ax = plt.subplots(figsize=(9, 8))
mn = min(loco_reg_df["DMI_NRC"].min(), loco_reg_df["DMI_est"].min()) - 1
mx = max(loco_reg_df["DMI_NRC"].max(), loco_reg_df["DMI_est"].max()) + 2
ax.plot([mn,mx],[mn,mx], "k--", lw=1.2, alpha=0.6, label="Perfect agreement")
ax.fill_between([mn,mx],[mn*0.9,mx*0.9],[mn*1.1,mx*1.1],
                alpha=0.07, color="green", label="±10% band")
for cow in all_cows:
    s = loco_reg_df[loco_reg_df["cow_id"]==cow]
    ax.scatter(s["DMI_NRC"], s["DMI_est"],
               color=COW_COLORS.get(cow,"gray"), s=55, alpha=0.8,
               label=f"Cow {cow}")
rmse_l = np.sqrt(((loco_reg_df["DMI_est"]-loco_reg_df["DMI_NRC"])**2).mean())
mape_l = (100*(loco_reg_df["DMI_est"]-loco_reg_df["DMI_NRC"]).abs()/loco_reg_df["DMI_NRC"]).mean()
r2_l   = r2_score(loco_reg_df["DMI_NRC"], loco_reg_df["DMI_est"])
ax.set_xlabel("NRC Reference DMI (kg/day)", fontsize=12)
ax.set_ylabel("LOCO-Predicted DMI (kg/day)\n(coefficients never trained on this cow)", fontsize=12)
ax.set_title(f"Leave-One-Cow-Out Cross-Validation\n"
             f"RMSE={rmse_l:.3f}  MAPE={mape_l:.1f}%  R²={r2_l:.3f}  n={len(loco_reg_df)}",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=7, ncol=2, framealpha=0.75)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,"2loco_scatter.png"),
            dpi=300, bbox_inches="tight")
plt.close()
print("Saved")


# Linear LOCO bias vs body weight


cow_bw   = [COW_BASE[c]["BW"] for c in all_cows]
cow_bias = [loco_lin_df[loco_lin_df["cow_id"]==c]["DMI_est"].values -
            loco_lin_df[loco_lin_df["cow_id"]==c]["DMI_NRC"].values
            for c in all_cows]
cow_bias_mean = [b.mean() for b in cow_bias]

fig, ax = plt.subplots(figsize=(9, 6))
for c, bw, bias in zip(all_cows, cow_bw, cow_bias_mean):
    ax.scatter(bw, bias, color=COW_COLORS.get(c,"gray"), s=90, zorder=4)
    ax.annotate(str(c), (bw, bias), fontsize=9,
                xytext=(4,4), textcoords="offset points")
ax.axhline(0, color="black", lw=1)
z = np.polyfit(cow_bw, cow_bias_mean, 1)
xline = np.linspace(min(cow_bw), max(cow_bw), 50)
ax.plot(xline, np.polyval(z, xline), "k--", lw=1.5, alpha=0.6,
        label=f"Trend: bias = {z[0]:.5f}·BW + {z[1]:.2f}")
corr = np.corrcoef(cow_bw, cow_bias_mean)[0,1]
ax.set_xlabel("Body Weight (kg)")
ax.set_ylabel("LOCO Mean Bias (kg/day)  [DMI_est − DMI_NRC]")
ax.set_title(f"Linear Model LOCO Bias vs Body Weight\n"
             f"r = {corr:.3f}  (n=16 cows)")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,"3linear_bias_bw.png"),
            dpi=300, bbox_inches="tight")
plt.close()
print("Saved")


# M-M scatter by camera

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, cam in zip(axes, ["001","002","003"]):
    sub = data[data["camera"]==cam]
    if sub.empty:
        ax.set_visible(False)
        continue
    mn = min(sub["DMI_NRC"].min(), sub["DMI_est_mm"].min()) - 1
    mx = max(sub["DMI_NRC"].max(), sub["DMI_est_mm"].max()) + 2
    ax.plot([mn,mx],[mn,mx], "k--", lw=1.2, alpha=0.6, label="Perfect")
    ax.fill_between([mn,mx],[mn*0.9,mx*0.9],[mn*1.1,mx*1.1],
                    alpha=0.07, color="green", label="±10%")
    for cow in sorted(sub["cow_id"].unique()):
        s = sub[sub["cow_id"]==cow]
        rmse_c = np.sqrt((s["err_mm"]**2).mean())
        ax.scatter(s["DMI_NRC"], s["DMI_est_mm"],
                   color=COW_COLORS.get(cow,"gray"), s=65, zorder=5,
                   label=f"Cow {cow}  RMSE={rmse_c:.2f}")
    s_rmse = np.sqrt((sub["err_mm"]**2).mean())
    s_mape = sub["err_mm"].abs().div(sub["DMI_NRC"]).mean()*100
    s_r2   = r2_score(sub["DMI_NRC"], sub["DMI_est_mm"])
    ax.set_xlabel("NRC Reference DMI (kg/day)")
    ax.set_ylabel("M-M Estimated DMI (kg/day)")
    ax.set_title(f"Camera {cam}\nRMSE={s_rmse:.3f}  MAPE={s_mape:.1f}%  R²={s_r2:.3f}  n={len(sub)}")
    ax.legend(fontsize=6.5, ncol=2, framealpha=0.7)
    ax.grid(alpha=0.3)
plt.suptitle(f"Michaelis-Menten Model by Camera (Km={KM_BEST} min/day, per-cow DMI_MAX)\n"
             f"Overall: RMSE={mm_whole['RMSE']}  MAPE={mm_whole['MAPE']}%  n={len(data)}",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,"4mm_by_camera.png"),
            dpi=300, bbox_inches="tight")
plt.close()
print("Saved")


# Welfare heatmap

def classify(ratio):
    if ratio < 0.9: return "underfed"
    elif ratio > 1.1: return "overfed"
    return "normal"

data["status_mm"] = data["ratio_mm"].apply(classify)

pivot = (data.pivot_table(index="cow_id", columns="day",
                          values="ratio_mm", aggfunc="mean")
         .reindex(columns=DAYS).reindex(all_cows))

cmap = LinearSegmentedColormap.from_list(
    "welfare", ["#E24B4A","#F5A623","#FFFFFF","#2ECC71","#3498DB"], N=512)

fig, ax = plt.subplots(figsize=(18, 10))
im = ax.imshow(pivot.values.astype(float), cmap=cmap, vmin=0.7, vmax=1.3, aspect="auto")
ax.set_xticks(range(len(DAYS)))
ax.set_xticklabels(DAY_LABELS, fontsize=11)
ax.set_yticks(range(len(all_cows)))
ax.set_yticklabels([f"Cow {c}  [{cam_map[c]}]" for c in all_cows], fontsize=10)
ax.set_xticks(np.arange(-0.5, len(DAYS), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(all_cows), 1), minor=True)
ax.grid(which="minor", color="white", lw=1.5)
ax.tick_params(which="minor", bottom=False, left=False)

for i, cow in enumerate(all_cows):
    for j, day in enumerate(DAYS):
        try:
            val = float(pivot.loc[cow, day])
            if not np.isnan(val):
                col = "white" if (val < 0.82 or val > 1.22) else "black"
                ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                        fontsize=8, color=col, fontweight="bold")
            else:
                ax.text(j, i, "—", ha="center", va="center",
                        fontsize=9, color="gray")
        except:
            ax.text(j, i, "—", ha="center", va="center",
                    fontsize=9, color="gray")

# Health event annotations
for cow, note, col in [(354,"Teat injury (Mar 20)","#C0392B"),
                       (349,"Milk fever (Mar 23)","#C0392B")]:
    if cow in all_cows:
        i = list(all_cows).index(cow)
        ax.annotate(note, xy=(10.5,i), xytext=(11.5,i), fontsize=16,
                    color=col, va="center", annotation_clip=False,
                    arrowprops=dict(arrowstyle="-", color=col, lw=0.8))

cbar = plt.colorbar(im, ax=ax, fraction=0.018, pad=0.01)
cbar.set_label("DMI Ratio  (Est / NRC)", fontsize=14)
cbar.ax.axhline(0.9, color="black", lw=1.5, ls="--")
cbar.ax.axhline(1.1, color="black", lw=1.5, ls="--")

patches = [mpatches.Patch(color="#E24B4A", label="Underfed (<0.9)"),
           mpatches.Patch(color="#2ECC71", label="Normal (0.9–1.1)"),
           mpatches.Patch(color="#3498DB", label="Overfed (>1.1)")]
ax.legend(handles=patches, loc="lower right", fontsize=14, framealpha=0.85)
ax.set_title(f"Feed Welfare Status — 16 Cows × 11 Days (March 2025)\n"
             f"M-M Model: per-cow DMI_MAX, Km={KM_BEST} min/day",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Day", fontsize=14)
ax.set_ylabel("Cow ID  [Camera]", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,"5welfare_heatmap.png"),
            dpi=300, bbox_inches="tight")
plt.close()
print("welfare_heatmap.png")


# Health event trajectories


HEALTH_EVENTS = {
    354: {"date":"March20", "label":"Teat injury recurrence (Anafen)"},
    349: {"date":"March23", "label":"Milk fever (IV calcium)"},
}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for col_idx, cow in enumerate([354, 349]):
    s = data[data["cow_id"]==cow].copy()
    s["day_idx"] = s["day"].apply(lambda d: DAYS.index(d))
    s = s.sort_values("day_idx").reset_index(drop=True)
    s["DMI_ratio"] = s["DMI_est_mm"] / s["DMI_NRC"]

    event_day = HEALTH_EVENTS[cow]["date"]
    event_idx = DAYS.index(event_day) if event_day in DAYS else None
    day_labels_cow = [d.replace("March","Mar ") for d in s["day"]]

    # Top panel — feeding time
    ax_top = axes[0, col_idx]
    ax_top.plot(s["day_idx"], s["feeding_min"], marker="o",
                color=COW_COLORS.get(cow,"steelblue"), lw=2, ms=7)
    if event_idx is not None:
        ax_top.axvline(event_idx, color="red", linestyle="--", alpha=0.7,
                       label=HEALTH_EVENTS[cow]["label"])
    ax_top.set_xticks(s["day_idx"])
    ax_top.set_xticklabels(day_labels_cow, rotation=45, ha="right", fontsize=10)
    ax_top.set_ylabel("Feeding duration (min/day)")
    ax_top.set_title(f"Cow {cow} — {HEALTH_EVENTS[cow]['label']}")
    ax_top.legend(fontsize=9)
    ax_top.grid(alpha=0.3)

    # Bottom panel — DMI ratio
    ax_bot = axes[1, col_idx]
    ax_bot.plot(s["day_idx"], s["DMI_ratio"], marker="o",
                color="darkorange", lw=2, ms=7, label="DMI ratio (M-M / NRC)")
    ax_bot.axhline(1.0, color="gray", linestyle=":", alpha=0.6, label="Target (1.0)")
    ax_bot.axhline(0.9, color="#E24B4A", linestyle="--", alpha=0.6,
                   lw=1.5, label="Alert threshold (0.9)")
    if event_idx is not None:
        ax_bot.axvline(event_idx, color="red", linestyle="--", alpha=0.7)
    ax_bot.set_xticks(s["day_idx"])
    ax_bot.set_xticklabels(day_labels_cow, rotation=45, ha="right", fontsize=10)
    ax_bot.set_ylabel("DMI ratio (est / NRC)")
    ax_bot.set_xlabel("Date")
    ax_bot.legend(fontsize=9)
    ax_bot.grid(alpha=0.3)

plt.suptitle("Welfare-Relevant Deviation Detection\n"
             "Camera-Derived Feeding Duration vs Independently Documented Health Events\n"
             "(Neither model received any health label as input)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,"6health_events.png"),
            dpi=300, bbox_inches="tight")
plt.close()
print("Saved")

# Print health event values for thesis
print(f"\n  Cow 354 detail:")
for _, row in data[data["cow_id"]==354].sort_values("DIM").iterrows():
    print(f"    {row['day']:>10}  feeding={row['feeding_min']:>6.1f}  "
          f"DMI_est={row['DMI_est_mm']:>6.2f}  DMI_NRC={row['DMI_NRC']:>6.2f}  "
          f"ratio={row['ratio_mm']:.3f}")


# Km sensitivity plot


fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(sweep_df["Km"], sweep_df["MAPE"], "o-", color="#3498DB", lw=2, ms=5)
axes[0].axvline(KM_BEST, color="red", ls="--", lw=2,
                label=f"Km={KM_BEST} (selected)")
axes[0].axhline(10, color="green", ls=":", lw=1.5, alpha=0.7,
                label="10% threshold")
axes[0].set_xlabel("Km (min/day)")
axes[0].set_ylabel("MAPE (%)")
axes[0].set_title("MAPE vs Km")
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

axes[1].plot(sweep_df["Km"], sweep_df["RMSE"], "o-", color="#1D9E75", lw=2, ms=5)
axes[1].axvline(KM_BEST, color="red", ls="--", lw=2)
axes[1].set_xlabel("Km (min/day)")
axes[1].set_ylabel("RMSE (kg DM/day)")
axes[1].set_title("RMSE vs Km")
axes[1].grid(alpha=0.3)

axes[2].plot(sweep_df["Km"], sweep_df["R2"], "o-", color="#9B59B6", lw=2, ms=5)
axes[2].axvline(KM_BEST, color="red", ls="--", lw=2)
axes[2].set_xlabel("Km (min/day)")
axes[2].set_ylabel("R²")
axes[2].set_title("R² vs Km")
axes[2].grid(alpha=0.3)

plt.suptitle(f"Km Sensitivity — Analytically Calibrated Per-Cow DMI_MAX\n"
             f"Km={KM_BEST} min/day selected (lowest MAPE in biological range 30–150 min/day)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,"7km_sensitivity.png"),
            dpi=300, bbox_inches="tight")
plt.close()
print("Saved")

print(f"\nAll Chapter 6 figures saved to: {FIG_DIR}")
print("Files:")
for f in sorted(os.listdir(FIG_DIR)):
    if f.endswith(".png"):
        print(f"  {f}")


Saved → fig6_1_regression_train_test.png
Saved → fig6_2_loco_scatter.png
Saved → fig6_3_linear_bias_bw.png
Saved → fig6_4_mm_by_camera.png
Saved → fig6_5_welfare_heatmap.png
Saved → fig6_6_health_events.png

  Cow 354 detail:
       March15  feeding= 257.8  DMI_est= 26.83  DMI_NRC= 27.00  ratio=0.994
       March16  feeding= 201.2  DMI_est= 26.06  DMI_NRC= 26.18  ratio=0.995
       March17  feeding= 235.2  DMI_est= 26.56  DMI_NRC= 25.41  ratio=1.045
       March18  feeding= 277.1  DMI_est= 27.02  DMI_NRC= 26.04  ratio=1.038
       March19  feeding=  75.6  DMI_est= 21.44  DMI_NRC= 26.34  ratio=0.814
Saved → fig6_7_km_sensitivity.png

All Chapter 6 figures saved to: /Users/shreyarao/Desktop/MooOutput/nutrition_results/
Files:
  fig6_1_regression_train_test.png
  fig6_2_loco_scatter.png
  fig6_3_linear_bias_bw.png
  fig6_4_mm_by_camera.png
  fig6_5_welfare_heatmap.png
  fig6_6_health_events.png
  fig6_7_km_sensitivity.png


In [6]:

def classify(ratio):
    if ratio < 0.9: return "underfed"
    elif ratio > 1.1: return "overfed"
    return "normal"

data["status_mm"] = data["ratio_mm"].apply(classify)

pivot = (data.pivot_table(index="cow_id", columns="day",
                          values="ratio_mm", aggfunc="mean")
         .reindex(columns=DAYS).reindex(all_cows))

cmap = LinearSegmentedColormap.from_list(
    "welfare", ["#E24B4A","#F5A623","#FFFFFF","#2ECC71","#3498DB"], N=512)

fig, ax = plt.subplots(figsize=(18, 10))
im = ax.imshow(pivot.values.astype(float), cmap=cmap, vmin=0.7, vmax=1.3, aspect="auto")
ax.set_xticks(range(len(DAYS)))
ax.set_xticklabels(DAY_LABELS, fontsize=14)
ax.set_yticks(range(len(all_cows)))
ax.set_yticklabels([f"Cow {c}  [{cam_map[c]}]" for c in all_cows], fontsize=14)
ax.set_xticks(np.arange(-0.5, len(DAYS), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(all_cows), 1), minor=True)
ax.grid(which="minor", color="white", lw=1.5)
ax.tick_params(which="minor", bottom=False, left=False)

for i, cow in enumerate(all_cows):
    for j, day in enumerate(DAYS):
        try:
            val = float(pivot.loc[cow, day])
            if not np.isnan(val):
                col = "white" if (val < 0.82 or val > 1.22) else "black"
                ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                        fontsize=16, color=col, fontweight="bold")
            else:
                ax.text(j, i, "—", ha="center", va="center",
                        fontsize=16, color="gray")
        except:
            ax.text(j, i, "—", ha="center", va="center",
                    fontsize=16, color="gray")

# Health event annotations
for cow, note, col in [(354,"Teat injury (Mar 20)","#C0392B"),
                       (349,"Milk fever (Mar 23)","#C0392B")]:
    if cow in all_cows:
        i = list(all_cows).index(cow)
        ax.annotate(note, xy=(10.5,i), xytext=(11.5,i), fontsize=16,
                    color=col, va="center", annotation_clip=False,
                    arrowprops=dict(arrowstyle="-", color=col, lw=0.8))

cbar = plt.colorbar(im, ax=ax, fraction=0.018, pad=0.01)
cbar.set_label("DMI Ratio  (Est / NRC)", fontsize=14)
cbar.ax.axhline(0.9, color="black", lw=1.5, ls="--")
cbar.ax.axhline(1.1, color="black", lw=1.5, ls="--")

patches = [mpatches.Patch(color="#E24B4A", label="Underfed (<0.9)"),
           mpatches.Patch(color="#2ECC71", label="Normal (0.9–1.1)"),
           mpatches.Patch(color="#3498DB", label="Overfed (>1.1)")]
ax.legend(handles=patches, loc="lower right", fontsize=14, framealpha=0.85)
ax.set_title(f"Feed Welfare Status\n",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Day", fontsize=14)
ax.set_ylabel("Cow ID  [Camera]", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,"new_fig6_5_welfare_heatmap.png"),
            dpi=300, bbox_inches="tight")
plt.close()
print("Saved → fig6_5_welfare_heatmap.png")

Saved → fig6_5_welfare_heatmap.png
